# Deploying LongCat Text-to-Image on SageMaker Async Inference

This notebook walks end-to-end through standing up an **asynchronous SageMaker endpoint** that serves the LongCat text-to-image model, then submits a prompt and downloads the rendered image. Async inference is the right fit here because diffusion models can take tens of seconds per request, which exceeds the 60-second hard limit on real-time endpoints.

**Flow at a glance:**
1. Pin the SageMaker SDK to a known-good version.
2. Package an empty `model.tar.gz` (model weights are pulled at container startup).
3. Upload the artifact to the default SageMaker S3 bucket.
4. Deploy a PyTorch model as an **async** endpoint with SNS notifications.
5. Submit a prompt via S3, poll for the output, and render the image.


## 1. Pin the SageMaker SDK

We uninstall any pre-existing `sagemaker` package and reinstall a `<3.0.0` build. The 2.x line still exposes `PyTorchModel`, `AsyncInferenceConfig`, and the deploy shorthand used below; 3.x reorganized several of these APIs and would require code changes elsewhere in the notebook.


In [ ]:
!pip uninstall -y sagemaker
!pip install "sagemaker<3.0.0"

## 2. Create a placeholder model artifact

SageMaker hosting requires a `model.tar.gz` to be referenced via `model_data`, even when the actual weights are downloaded at container start time (in this case, pulled from Hugging Face using `HF_TOKEN`). Writing an **empty gzip tarball** satisfies the API contract without shipping any real weights through S3 — keeping the artifact tiny and the upload near-instant.


In [ ]:
import tarfile

with tarfile.open("model.tar.gz", "w:gz") as tar:
    pass  # empty archive

## 3. Upload the artifact to S3

`Session.upload_data` pushes the (empty) tarball to the account's default SageMaker bucket under the `longcat-dummy/` prefix and returns the resulting `s3://...` URI. This URI feeds directly into `PyTorchModel(model_data=...)` in the next step.


In [ ]:
import sagemaker

sess = sagemaker.Session()
bucket = sess.default_bucket()

s3_path = sess.upload_data("model.tar.gz", bucket=bucket, key_prefix="longcat-dummy")

print(s3_path)

## 4. Deploy as an asynchronous endpoint

This is the heart of the notebook. A few things worth calling out:

- **`AsyncInferenceConfig`** redirects results to `s3://.../longcat-outputs/` and failures to `s3://.../longcat-failures/`, with SNS notifications on both success and error so downstream services can react without polling.
- **`max_concurrent_invocations_per_instance=1`** serializes requests on each GPU. Diffusion inference is VRAM-bound; allowing concurrent requests on a single instance would OOM the model.
- **`entry_point="inference.py"` + `source_dir="code"`** points SageMaker at the custom handler that loads the diffusers pipeline and renders prompts.
- **`ml.g6e.2xlarge`** gives an L40S GPU (48 GB VRAM) — enough headroom for 1344×768 generations at fp16/bf16.

The trailing `-----------!` in the output is SageMaker's progress indicator while the endpoint comes up; the `!` signals it transitioned to `InService`.


In [ ]:
from sagemaker.pytorch import PyTorchModel
import sagemaker
from sagemaker.async_inference import AsyncInferenceConfig
import os
import time

sess = sagemaker.Session()
bucket = sess.default_bucket()
role = sagemaker.get_execution_role()


sns_topic_arn = os.environ["LONGCAT_SNS_TOPIC_ARN"]  # e.g. arn:aws:sns:<region>:<account-id>:<topic-name>

async_config = AsyncInferenceConfig(
    output_path=f"s3://{bucket}/longcat-outputs/",
    failure_path=f"s3://{bucket}/longcat-failures/",
    notification_config={
        "SuccessTopic": sns_topic_arn,  # ← notified when inference succeeds
        "ErrorTopic":   sns_topic_arn,  # ← notified when inference fails
    },
    max_concurrent_invocations_per_instance=1,
)

role = sagemaker.get_execution_role()

endpoint_name = f"longCat-text-to-image"

model = PyTorchModel(
    role=role,
    model_data=s3_path,  # 👈 REQUIRED FIX
    entry_point="inference.py",
    source_dir="code",
    framework_version="2.5",
    py_version="py311",
    env={
        "HF_TOKEN": os.environ["HF_TOKEN"]  # export before running, e.g. via Secrets Manager
    }
)


predictor = model.deploy(
    endpoint_name=endpoint_name,
    initial_instance_count=1,
    instance_type="ml.g6e.2xlarge",
    async_inference_config=async_config
)

##### 5. Upload the prompt payload to S3

Async inference does **not** accept the request body inline; the payload must live in S3 and be referenced by URI. We serialize the diffusion parameters (`inputs`, `negative_prompt`, `guidance_scale`, `num_inference_steps`, and the target 1344×768 resolution) to JSON and `PutObject` it under `longcat-inputs/`. The returned `s3://...` path is what `invoke_endpoint_async` consumes next.


In [ ]:
import boto3, json

s3 = boto3.client("s3")

# Upload your input to S3
input_data = json.dumps({
    "inputs": """
    Oil painting of a resilient female pirate captain standing on a bustling wooden dock, her weathered skin featuring a scar on her left breast, with a determined gaze. Her hair is intricately braided with small, found objects that catch the light. She wears a practical yet stylish outfit: a white blouse and maroon pants crafted from repurposed leather and salvaged materials. The scene uses a wide angle with strong backlighting, capturing vast scale and resilience. Ocean waves splash against the pilings, and heat haze distorts the air around her. The wooden planks underfoot show deep cracks, bearing the weight of time. In the distance, a sloop sails on the shimmering water. The mood is melancholic yet hopeful, with dramatic lighting emphasizing both the character's strength and the harsh beauty of the environment.""",
    "negative_prompt": "blurry, low quality, watermark, text",
    "guidance_scale": 4.5,
    "num_inference_steps": 40,
    "height": 768,
    "width": 1344
})
input_key = f"longcat-inputs/request-{int(time.time())}.json"

s3.put_object(
    Bucket=bucket,
    Key=input_key,
    Body=input_data.encode("utf-8"),
    ContentType="application/json"
)

input_s3_path = f"s3://{bucket}/{input_key}"
print(f"Input uploaded to: {input_s3_path}")

## 6. Fire the async invocation

`invoke_endpoint_async` returns **immediately** with an `OutputLocation` pointing at the S3 key where the response will land once inference finishes. There's no streaming and no blocking — the request is queued internally by SageMaker and dispatched to a worker as capacity allows.


In [ ]:
import boto3, json, time

sm_runtime = boto3.client("sagemaker-runtime")

# Send the request — returns immediately, doesn't wait
response = sm_runtime.invoke_endpoint_async(
    EndpointName=predictor.endpoint_name,
    InputLocation=input_s3_path,   # ← input must be in S3 (see below)
    ContentType="application/json",
)

output_location = response["OutputLocation"]
print(f"Output will appear at: {output_location}")

## 7. Poll S3 for the rendered image

Until the worker writes the output, `GetObject` raises `NoSuchKey`. We loop with a 2-second sleep, treating `NoSuchKey` as "still running" and any successful read as "done". For a real application you'd subscribe to the `SuccessTopic` SNS topic configured above instead of polling — but for an interactive notebook polling is the simplest thing that works.

Each `Still running...` line corresponds to roughly two seconds of wall time; the run shown here finished in ~28 seconds.


In [ ]:
import boto3, time

s3 = boto3.client("s3")

# Parse the output location you got from invoke_endpoint_async
output_location = response["OutputLocation"]  # e.g. s3://bucket/longcat-outputs/xxx.out
path = output_location.replace("s3://", "")
out_bucket, out_key = path.split("/", 1)

while True:
    try:
        obj = s3.get_object(Bucket=out_bucket, Key=out_key)
        print("Done! Downloading result...")
        image_bytes = obj["Body"].read()
        break
    except s3.exceptions.NoSuchKey:
        print("Still running...")
        time.sleep(2)


## 8. Decode and display the result

The async handler returned raw image bytes. We decode them with PIL, downscale to a 400-px-tall preview using **Lanczos** resampling (highest-quality downsampler in PIL), and render inline. The full-resolution image is still available in `image` if you want to save it.


In [ ]:
from PIL import Image
from IPython.display import display
import io

image = Image.open(io.BytesIO(image_bytes))

# Resize to fit screen height (adjust 400 to your screen)
target_height = 400
ratio = target_height / image.height
preview = image.resize((int(image.width * ratio), target_height), Image.LANCZOS)
display(preview)
print(f"Original: {image.width}x{image.height} | Preview: {preview.width}x{preview.height}")


## Next steps

Scratch cell for follow-up experiments — try varying `guidance_scale`, swapping in a different prompt, or wiring the SNS topic up to a Lambda that pushes results into your application.
